# 03 - Avaliação do modelo e métricas

Este notebook documenta a avaliação final dos modelos e a interpretação das métricas usadas no projeto.

A avaliação oficial é executada pelo script:

```bash
poetry run python scripts/evaluate_model.py
```

O script gera arquivos em `reports/`. O dashboard lê esses arquivos para exibir a qualidade do modelo e escolher automaticamente o modelo ativo.

## 1. Preparação

Vamos carregar os relatórios já gerados pelo benchmark. Se você quiser recalcular tudo, existe uma célula opcional mais abaixo.

In [ ]:
import json
import subprocess
import sys
from pathlib import Path

import pandas as pd
import plotly.express as px

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

REPORTS_DIR = PROJECT_ROOT / "reports"
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

REPORTS_DIR

## 2. Opcional: recalcular o benchmark

Por padrão, a célula abaixo não roda. Ela pode demorar porque todos os modelos são avaliados na base real.

Para recalcular os relatórios, troque `RUN_BACKTEST` para `True`.

In [ ]:
RUN_BACKTEST = False

if RUN_BACKTEST:
    subprocess.run(
        [sys.executable, str(PROJECT_ROOT / "scripts" / "evaluate_model.py")],
        cwd=PROJECT_ROOT,
        check=True,
    )

## 3. Carregar relatórios

Arquivos principais:

- `model_metrics.json`: resumo do modelo vencedor.
- `model_comparison.csv`: comparação de todos os modelos.
- `model_metrics_by_family.csv`: métricas do vencedor por família de produto.

In [ ]:
required_reports = [
    "model_metrics.json",
    "model_comparison.csv",
    "model_metrics_by_family.csv",
]

missing_reports = [name for name in required_reports if not (REPORTS_DIR / name).exists()]
if missing_reports:
    raise FileNotFoundError(
        f"Relatórios ausentes em {REPORTS_DIR}: {missing_reports}. "
        "Execute poetry run python scripts/evaluate_model.py."
    )

with open(REPORTS_DIR / "model_metrics.json", encoding="utf-8") as file:
    winner_metrics = json.load(file)

comparison = pd.read_csv(REPORTS_DIR / "model_comparison.csv")
by_family = pd.read_csv(REPORTS_DIR / "model_metrics_by_family.csv")

winner_metrics

## 4. Definição das métricas

O projeto usa um conjunto de métricas complementares:

- **MAE**: erro absoluto médio. Fácil de interpretar, mas não pondera o peso total do volume.
- **RMSE**: penaliza erros grandes com mais intensidade.
- **RMSLE**: compara erro em escala logarítmica, reduzindo impacto de séries muito grandes.
- **SMAPE**: erro percentual simétrico, útil quando há grande variação de escala.
- **WAPE**: erro absoluto total dividido pelo volume real total. É a métrica principal do projeto.
- **Bias**: indica tendência de superestimar ou subestimar.

A escolha do modelo vencedor usa **menor WAPE**, porque o dashboard apoia planejamento agregado de vendas. Em outras palavras: errar muito em uma frente de alto volume pesa mais que errar na mesma proporção em uma frente pequena.

In [ ]:
metric_columns = ["mae", "rmse", "rmsle", "smape", "wape", "bias"]

comparison_display = comparison[
    ["model", "model_role", "validation_days", "rows_scored", "series_scored", *metric_columns]
].sort_values("wape")

comparison_display

In [ ]:
formatted_comparison = comparison_display.copy()
for column in ["smape", "wape", "bias"]:
    formatted_comparison[column] = formatted_comparison[column].map(lambda value: f"{value:.2%}")

formatted_comparison

## 5. Visualizar comparação dos modelos

A visualização abaixo mostra o critério de seleção principal: menor WAPE.

In [ ]:
fig = px.bar(
    comparison.sort_values("wape", ascending=False),
    x="wape",
    y="model",
    color="model_role",
    orientation="h",
    text=comparison.sort_values("wape", ascending=False)["wape"].map(lambda value: f"{value:.1%}"),
    title="WAPE por modelo avaliado",
    labels={"wape": "WAPE", "model": "modelo", "model_role": "tipo"},
)
fig.update_xaxes(tickformat=".0%")
fig.update_traces(textposition="outside")
fig.show()

## 6. Modelo vencedor

O arquivo `model_metrics.json` guarda o resultado final usado pelo dashboard.

Além das métricas, ele registra:

- janela de validação;
- quantidade de linhas avaliadas;
- quantidade de séries loja-família;
- baseline principal;
- ganho de WAPE contra o baseline principal;
- lista de modelos testados.

In [ ]:
winner_summary = pd.Series(
    {
        "modelo_vencedor": winner_metrics["model"],
        "criterio": winner_metrics["selected_by"],
        "validacao_inicio": winner_metrics["validation_start"],
        "validacao_fim": winner_metrics["validation_end"],
        "linhas_avaliadas": winner_metrics["rows_scored"],
        "series_avaliadas": winner_metrics["series_scored"],
        "wape": f"{winner_metrics['wape']:.2%}",
        "smape": f"{winner_metrics['smape']:.2%}",
        "bias": f"{winner_metrics['bias']:.2%}",
        "baseline_principal": winner_metrics["baseline_model"],
        "ganho_wape_vs_baseline": f"{winner_metrics['wape_lift_vs_baseline']:.2%}",
    }
)
winner_summary

## 7. Métricas por família de produto

A métrica geral é importante, mas pode esconder variações relevantes por família.

A tabela por família ajuda a responder:

- onde o modelo performa melhor;
- onde o erro percentual é maior;
- quais famílias merecem modelagem especializada no futuro.

In [ ]:
by_family.sort_values("wape").head(10)

In [ ]:
by_family.sort_values("wape", ascending=False).head(10)

In [ ]:
fig = px.bar(
    by_family.sort_values("wape", ascending=False).head(15).sort_values("wape"),
    x="wape",
    y="family",
    orientation="h",
    title="Top 15 famílias com maior WAPE",
    labels={"wape": "WAPE", "family": "família"},
)
fig.update_xaxes(tickformat=".0%")
fig.show()

## 8. Interpretação do Bias

O `Bias` mede tendência direcional:

- Bias positivo: o modelo tende a prever acima do real.
- Bias negativo: o modelo tende a prever abaixo do real.
- Bias próximo de zero: os erros positivos e negativos se compensam no agregado.

Bias baixo não significa modelo perfeito. Um modelo pode compensar erros grandes em direções opostas. Por isso ele deve ser lido junto com WAPE, MAE e RMSE.

In [ ]:
fig = px.bar(
    comparison.sort_values("bias"),
    x="bias",
    y="model",
    color="model_role",
    orientation="h",
    title="Bias por modelo",
    labels={"bias": "Bias", "model": "modelo", "model_role": "tipo"},
)
fig.update_xaxes(tickformat=".0%")
fig.add_vline(x=0, line_dash="dot")
fig.show()

## 9. Como o dashboard usa essa avaliação

A integração acontece em duas partes:

1. `scripts/evaluate_model.py` gera `reports/model_metrics.json` com o modelo vencedor.
2. `src/revenue_forecast_dashboard/data.py` lê esse arquivo e instancia o modelo pelo nome.

Também é possível forçar outro modelo pela variável de ambiente:

```bash
FORECAST_MODEL_NAME=SeasonalPromotionTrendForecaster
```

Isso permite testar alternativas sem alterar o código do dashboard.

In [ ]:
print("Modelo ativo no relatório:", winner_metrics["model"])
print("Modelos avaliados:")
for model_name in winner_metrics["benchmark_models"]:
    print("-", model_name)

## 10. Limitações e próximos passos

Limitações atuais:

- A base não possui preço, margem ou receita financeira real.
- O modelo vencedor é escolhido por uma única janela final de validação.
- Feriados, óleo e transações ainda não entram diretamente no modelo atual.
- Não há modelos especializados por loja, cluster ou família.

Evoluções recomendadas:

1. Criar validação temporal com múltiplas janelas.
2. Testar features de feriados nacionais, locais e eventos.
3. Incorporar transações como proxy de fluxo de loja.
4. Testar modelos por família ou cluster de loja.
5. Adicionar tabela de preço/ticket médio para converter vendas em receita monetária.
6. Monitorar drift de erro após cada nova atualização dos dados.

Com isso, o projeto fica preparado para evoluir de um baseline executivo confiável para um sistema de previsão mais robusto.